# Exploration of Making a Complete Reactor Mesh with MOOSE
Using the example of a 15x15 cylindrical BWR core

## Goal
Explore how to use the MOOSE reactor module's mesh builder to create a model of a BWR reactor core, usable by OPENMC, NEKRS, and MOOSE

### Learning Objectives
1. Understand how to create different core elements

2. Learn how to break down a complex mesh into workable pieces

3. Learn how to account for differing mesh element types

4. Learn how to Prepare a mesh for use by cardinal solvers, such as NEKRS, OPENMC, and other MOOSE modules

## Functions
Execute the following code cell to declare any needed functions and global variables

In [ ]:
import subprocess as sp
import tempfile as tp
import os
import pyvista as pv



Text_sections=[]
def gather_sectors(tlist):
  otext="".join(Text_sections)
  return otext
def run_cardinal_file(code):
  oname="workingoutput.e"
  with tp.TemporaryDirectory(ignore_cleanup_errors=True) as work_dir:
    ifileloc = os.path.join(work_dir,"inputfile.i")
    ofileloc = os.path.join(work_dir, oname)

    with open(ifileloc,"w") as ifile:
      ifile.write(code)

    sp.run(['cardinal-opt','-i',ifileloc,'--mesh-only',ofileloc],check=True)

    ofilereader=pv.get_reader(ofileloc)
    ofile=ofilereader.read()
    ofile.plot(show_edges=True, jupyter_backend='static')
def run_code():
  combotxt=gather_sectors(Text_sections)
  run_cardinal_file(combotxt)


# Overview
## Context
We are creating the mesh of a BWR, which means that we need to construct a core consisting of fuel bundles and control rods. Fuel bundles consist of a set of fuel pins, or vertical cylinders of fuel surrounded by cladding, and water pins, or hollow pins made of cladding to allow water to better traverse the middle of the bundles. This mesh is more simplistic, so we are not implementing the "proper" fuel bundle structure with holder plates/structures, just the pins.

## Layout
We will break this mesh down into distinct sections, 
### 1. Setup substitution variables
a. Setup constants

b. Setup derived variables

### 2. Create the Cruciform Control Rods
a. Create the control rod sub sections for each of construction with simple shapes

b. Merge these subsections in one combined Cruciform_Mesh


### 3. Create the pins for the bundle
a. Create Fuel Pins

b. Create the Water Pins


### 4. Create the Fuel bundles
a. Create the Bundle Mesh

b. Fix Mesh element types


### 5. Create fuel assemblies 
a. Create the quad bundle assembly

b. Create the single bundle assemblies

c. Create the double bundle assemblies


### 6. Create Core Assembly
a. Create the rough core mesh

b. Clear all dummy elements from core

c. Repair mesh boundaries for use by Cardinal

d. Add core periphery

e. Finish cleaning boundaries for Cardinal use

# 1. Setting up substitution variables

### Setting Basic Constants and Dimensions
We will need to define several very important basic dimensions of our reactor, such as how big a fuel pin is, how thick the cladding is, etc.


In [ ]:
constants="""
# Control Rod Dimensions

#Rcr is the half-width of the control rod arms
rcr = 1.5

#W1 is the width of the enclosing square of a pin
wpin = 5.0

#W2 is the width of one square fuel bundle, which is an 8x8 pin square
wbundle = '{fparse wpin*8}'

#Rbundle is the half-width a fuel-bundle
rbundle = '{fparse wbundle*0.5}'

#L1 is the length of the rod from the center
#L1 is ideally 2rcr + 6*wpin so it ends up roughly 2/3 of the fuel bundles,
#as there should be rcr between the fuel bundles and the control rod,
# so add two rcr and 6 wpin
l1 = '{fparse 2*rcr+6*wpin}'

#W2 is the width of the Bundle Assembly meshes
#It should be the width of two fuel bundles, plus the width of the control rod arm, which is 2*rcr, plus the gap between the control rod and bundles, which is 2*rcr, plus 1 unit of headroom around the outside, so +2
w2 = '{fparse 2*wbundle + 4*rcr + 2}'

#Rassem is the half-width of the bundle assembly submeshes
Rsubassem = '{fparse w2*05}'







 """

# 2. Making the Control Rod Mesh

## Approach
1. Create center of cross shape
2. Create horizontal sections
3. Join horizontal sections to center
4. Create vertical sections
5. Create final control rod mesh